# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 26.1367


In [2]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.72 GB
MemFree: 16.46 GB
MemAvailable: 909.98 GB
Free GPU Memory (GB): 26.1367

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################



/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful
Successfully authenticated with the Hugging Face API.

################################
Setting up GPU memory usage list...
################################

Free GPU Memory (GB): 26.1367. Context: Warm up notebook.


## 3. SEML Pipeline

In [3]:
# Defining Pipeline
import tempfile
from src.models import get_model, get_model_name, get_tokenizer
from src.data import get_dataset, data_loader_from_split
from src.algorithms.quantization.quantize import quantize
from src.evaluations.evaluate_all import evaluate

from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}

def run_quantize(
    # Dataset parameters
    seed_dataset=123,
    directory_dataset="",
    calib_dataset_name="",
    calib_dataset_split="",
    calib_seq_length=2048,
    eval_dataset_name="",
    eval_dataset_split="",
    eval_seq_length=2048,
    eval_n_samples=None,
    batch_size=1,
    # Model parameters
    seed_model=123,
    directory_model="",
    clean_cache=True,
    model_name="",
    # Quantization parameters
    quantize_method="NONE",
    # Evaluation metrics
    eval_metrics=[
        "perplexity",
        "brier_score",
        "disk_space_usage",
        "quantize_runtime"
    ],
    device="cuda",
    save_quantized_model=False,
    quantized_model_save_path="",
):
    ##################
    ## Print config ##
    ##################
    print("Received the following configuration:")
    print(
        f"Calibration dataset: {calib_dataset_name}\n"
        f"Calibration split: {calib_dataset_split}\n"
        f"Calibration sequence length: {calib_seq_length}\n"
        f"Evaluation dataset: {eval_dataset_name}\n"
        f"Evaluation split: {eval_dataset_split}\n"
        f"Evaluation sequence length: {eval_seq_length}\n"
        f"Evaluation number of samples: {eval_n_samples}\n"
        f"Batch size: {batch_size}\n"
        f"Model: {model_name}\n"
        f"Quantize method: {quantize_method}\n"
        f"Evaluation metrics: {eval_metrics}\n"
        f"Device: {device}\n"
        f"Quantized model save: {save_quantized_model}\n"
        f"Quantized model save path: {quantized_model_save_path}\n"
    )
    
    with tempfile.TemporaryDirectory(prefix=CACHE_PATH) as temp_cache_dir:
        print(f"Setting cache path to {temp_cache_dir}")
        os.environ["TORCH_HOME"] = temp_cache_dir
        os.environ["HF_HOME"] = temp_cache_dir
        os.environ["HUGGINGFACE_HUB_CACHE"] = temp_cache_dir
        os.environ["HUGGINGFACE_ASSETS_CACHE"] = temp_cache_dir
        os.environ["TRANSFORMERS_CACHE"] = temp_cache_dir
        torch.hub.set_dir(temp_cache_dir)
            
        ####################
        ## Load tokenizer ##
        ####################
        print("Load tokenizer")
        model_full_name = get_model_name(model_name)
        tokenizer = get_tokenizer(
            model_name=model_full_name,
            seed=seed_model,
            directory_model=directory_model,
            device=device
        )
        print(f"Loaded tokenizer: {model_full_name}")
        record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Load tokenizer")
        
        ###############
        ## Load data ##
        ###############
        print("Load calibration data module")
        calib_data_module = get_dataset(
            dataset_name=calib_dataset_name,
            directory_dataset=directory_dataset,
            batch_size=batch_size,
            sequence_length=calib_seq_length,
            tokenizer_name=model_full_name,
            seed=seed_dataset,
        )
        print("Load evaluation data module")
        eval_data_module = get_dataset(
            dataset_name=eval_dataset_name,
            directory_dataset=directory_dataset,
            batch_size=batch_size,
            sequence_length=eval_seq_length,
            tokenizer_name=model_full_name,
            seed=seed_dataset,
        )
        
        print("Load calibration dataloader")
        calib_dataloader = data_loader_from_split(
            data_module=calib_data_module,
            split=calib_dataset_split,
            sequence_length=512 if quantize_method in ["AWQ-4"] else calib_seq_length,
        )
        print("Load evaluation dataloader")
        eval_dataloader = data_loader_from_split(
            data_module=eval_data_module,
            split=eval_dataset_split,
            sequence_length=512 if quantize_method in ["AWQ-4"] else eval_seq_length,
        )
        record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Load data")

        ##############
        ## Quantize ##
        ##############
        print("Quantization...")
        quantized_model = None
        if quantize_method == "NONE":
            print("Quantize method is None, loading original model")
            quantized_model = get_model(
                model_name=model_full_name,
                seed=seed_model,
                directory_model=directory_model,
                device=device,
            )
            record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Load base model")
        else:
            print(f"Quantizing model using {quantize_method}")
            quantized_model = quantize(
                model_name=model_full_name,
                tokenizer=tokenizer,
                calib_dataloader=calib_dataloader,
                train_dataloader=calib_dataloader,
                quantize_method=quantize_method,
                save_model=save_quantized_model,
                save_path=quantized_model_save_path,
                device=device
            )
            record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Quantize model")

        ##############
        ## Evaluate ##
        ##############
        print("Evaluation...")
        results = evaluate(
            model=quantized_model,
            eval_dataloader=eval_dataloader,
            eval_metrics=eval_metrics,
            n_samples=eval_n_samples,
            device=device,
            to_device=("AWQ" in quantize_method),
            prefix="",
            gpu_memory_usage=gpu_memory_usage
        )
        record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Evaluate model")

    fail_trace = {
        "fail_trace": seml.evaluation.get_results,
    }

    return {**results, **fail_trace}

In [4]:
import itertools
import random
import torch

from src.algorithms.quantization import QUANT_CONFIGS  # Ensure torch is imported

# Fixed parameters
fixed_params = {
    'device': 'cuda',
    'clean_cache': True,
    'save_quantized_model': True,
    'seed_model': 123,
    'seed_dataset': 123,
    'batch_size': 1,
    'eval_metrics':
        [
            # 'perplexity',
            # 'brier_score',
            'disk_space_usage',
            'gpu_utilization'
        ],
    'calib_dataset_split': 'validation',
    'eval_dataset_split': 'test',
    'calib_seq_length': 2048,
    'eval_seq_length': 2048,
    'eval_n_samples': 128
}

# Grid parameters
grid_params = {
    'calib_dataset_name': ['C4'],
    'eval_dataset_name': ['WikiText'],
    # 'quantize_method': ['AQLM-PREQUANTIZED', 'AQLM-LORA', 'HQQ-mixed', 'HQQ-LORA', 'QUANTO', 'AWQ-4', 'BNB-8', 'NONE'],
    'quantize_method': list(QUANT_CONFIGS.keys()),
    'model_name': ['TinyLlama']
}

batch_sizes = [1]

# Generate all combinations for grid search
grid_combinations = list(itertools.product(
    grid_params['calib_dataset_name'],
    grid_params['eval_dataset_name'],
    grid_params['quantize_method'],
    grid_params['model_name']
))

# Run the quantize function for all combinations
results = []
max_combinations = 10000  # Set to a lower number for testing purposes
for i, combination in enumerate(grid_combinations):
    if i >= max_combinations:
        break
    for batch_size in batch_sizes:
        calib_dataset_name, eval_dataset_name, quantize_method, model_name = combination

        # Print current combination details
        print(f"Running combination {i+1}/{len(grid_combinations)}")
        print(f"  Model Name: {model_name}")
        print(f"  Calibration Dataset: {calib_dataset_name}")
        print(f"  Evaluation Dataset: {eval_dataset_name}")
        print(f"  Quantize Method: {quantize_method}")
        print(f"  Batch Size: {batch_size}")

        result = run_quantize(
            # Fixed parameters
            device=fixed_params['device'],
            clean_cache=fixed_params['clean_cache'],
            save_quantized_model=fixed_params['save_quantized_model'],
            seed_model=fixed_params['seed_model'],
            seed_dataset=fixed_params['seed_dataset'],
            eval_metrics=fixed_params['eval_metrics'],
            calib_dataset_split=fixed_params['calib_dataset_split'],
            eval_dataset_split=fixed_params['eval_dataset_split'],
            calib_seq_length=fixed_params['calib_seq_length'],
            eval_seq_length=fixed_params['eval_seq_length'],
            eval_n_samples=fixed_params['eval_n_samples'],
            # Grid parameters
            calib_dataset_name=calib_dataset_name,
            eval_dataset_name=eval_dataset_name,
            quantize_method=quantize_method,
            model_name=model_name,
            # Random parameters
            batch_size=batch_size,
            # Model parameters
            directory_model="",
            directory_dataset="",
            quantized_model_save_path=""
        )

        # Append result with parameter details
        results.append({
            'result': result,
            'parameters': {
                'model_name': model_name,
                'calib_dataset_name': calib_dataset_name,
                'eval_dataset_name': eval_dataset_name,
                'calib_dataset_split': fixed_params['calib_dataset_split'],
                'eval_dataset_split': fixed_params['eval_dataset_split'],
                'calib_seq_length': fixed_params['calib_seq_length'],
                'eval_seq_length': fixed_params['eval_seq_length'],
                'eval_n_samples': fixed_params['eval_n_samples'],
                'quantize_method': quantize_method,
                'batch_size': batch_size,
            }
        })

# Do something with the results
print(results)

Running combination 1/12
  Model Name: TinyLlama
  Calibration Dataset: C4
  Evaluation Dataset: WikiText
  Quantize Method: NONE
  Batch Size: 1
Received the following configuration:
Calibration dataset: C4
Calibration split: validation
Calibration sequence length: 2048
Evaluation dataset: WikiText
Evaluation split: test
Evaluation sequence length: 2048
Evaluation number of samples: 128
Batch size: 1
Model: TinyLlama
Quantize method: NONE
Evaluation metrics: ['disk_space_usage', 'gpu_utilization']
Device: cuda
Quantized model save: True
Quantized model save path: 

Setting cache path to /nfs/students/daro/.cache/huggingfacemil6l2v2
Load tokenizer
Loaded tokenizer: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Free GPU Memory (GB): 26.1367. Context: Load tokenizer.
Load calibration data module
Load evaluation data module
Load calibration dataloader


Token indices sequence length is longer than the specified maximum sequence length for this model (1073002 > 2048). Running this sequence through the model will result in indexing errors


Load evaluation dataloader


Token indices sequence length is longer than the specified maximum sequence length for this model (341469 > 2048). Running this sequence through the model will result in indexing errors


Free GPU Memory (GB): 26.1367. Context: Load data.
Quantization...
Quantize method is None, loading original model


...

trainable params: 5,505,024 || all params: 2,047,676,416 || trainable%: 0.2688


max_steps is given, it will override any value given in num_train_epochs
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/torch/utils/cpp_extension.py:1967: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/torch/utils/checkpoint.py:464: Use

Step,Training Loss
1,3.583100
2,1.939200
3,2.585600
4,1.955400
5,2.172300
6,2.782800
7,2.365300
8,2.569600
9,2.968100
10,2.116700


/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Free GPU Memory (GB): 20.4062. Context: Quantize model.
Evaluation...
Free GPU Memory (GB): 20.4062. Context: Evaluate GPU type.
Free GPU Memory (GB): 20.4062. Context: Evaluate disk space usage.
Free GPU Memory (GB): 20.4062. Context: Evaluate model.
[{'result': {'current_gpu_type': 'NVIDIA A100-PCIE-40GB', 'current_gpu_total_memory': 40339.3125, 'current_gpu_free_memory': 23.587890625, 'disk_space_usage': 'nan', 'fail_trace': <function get_results at 0x7fd22e984160>}, 'parameters': {'model_name': 'TinyLlama', 'calib_dataset_name': 'C4', 'eval_dataset_name': 'WikiText', 'calib_dataset_split': 'validation', 'eval_dataset_split': 'test', 'calib_seq_length': 2048, 'eval_seq_length': 2048, 'eval_n_samples': 128, 'quantize_method': 'NONE', 'batch_size': 1}}, {'result': {'current_gpu_type': 'NVIDIA A100-PCIE-40GB', 'current_gpu_total_memory': 40339.3125, 'current_gpu_free_memory': 24.60546875, 'disk_space_usage': '1020.20 MB', 'fail_trace': <function get_results at 0x7fd22e984160>}, 'parame

In [5]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Free GPU Memory (GB): 0.0146484


In [7]:
[result['result']['quantize_runtime'] for result in results]

KeyError: 'quantize_runtime'